# Team-Level Market Value Regression -- Practice Notebook

This is a **practice notebook**, separate from the main assignment deliverable
(`FIFA_World_Cup_Dashboard_Assignment.ipynb`). The goal here isn't a finished dashboard -- it's practicing the
scikit-learn regression workflow: `train_test_split`, fitting a `LinearRegression`, and evaluating it.

**How this notebook is organized:**
1. Sections 1-3 (data load, team-table build, correlation/VIF check) are **fully worked** -- read through them,
   they set up the clean feature table you'll regress on.
2. Section 4 onward (splitting into two feature sets, and the actual regression) has **blanks (`____`)** for
   you to fill in yourself. Those cells will raise a `SyntaxError` or `NameError` until you replace the blanks --
   that's expected. Explanatory markdown above each blank tells you exactly what belongs there and why.

**Dependencies:** `pandas`, `numpy`, `statsmodels`, `scikit-learn`.


## 1. Setup and Load Data

In [16]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

df = pd.read_csv('fifa_world_cup_2026_player_performance.csv')
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


Loaded 54,600 rows x 75 columns


## 2. Build the Team-Level Table -- Using RATES, Not Sums

**Why rates, not sums?** If you aggregate raw match stats with `.sum()`, every team's total scales with how
many matches that team's players collectively played. In this dataset, teams range from 33 to 66 rows per
player, and that single "how much did this team play" factor correlates at r=0.98-0.99 with almost every
summed stat (total passes, total shots, total tackles, ...). That means a sum-based team table is riddled with
fake multicollinearity that has nothing to do with football skill -- it's just measuring playing time over and
over under different column names.

**The fix:** aggregate with `.mean()` instead -- the *average per player-match row* -- which is not affected by
how many matches a team happened to play. We verify below that this actually fixes the problem before
moving on.

In [17]:
# Demonstration: sums are driven by a hidden "how many matches played" confound
team_minutes = df.groupby('team')['minutes_played'].sum().rename('total_minutes')
team_shots_sum = df.groupby('team')['shots'].sum().rename('shots_sum')
team_passes_sum = df.groupby('team')['total_passes'].sum().rename('passes_sum')

confound_check = pd.concat([team_minutes, team_shots_sum, team_passes_sum], axis=1)
print("Correlation matrix -- notice how strongly total_minutes correlates with EVERY summed stat:")
print(confound_check.corr().round(3))


Correlation matrix -- notice how strongly total_minutes correlates with EVERY summed stat:
               total_minutes  shots_sum  passes_sum
total_minutes          1.000      0.896       0.958
shots_sum              0.896      1.000       0.937
passes_sum             0.958      0.937       1.000


In [18]:
rate_cols = ['goals','assists','shots','shots_on_target','expected_goals_xg','expected_assists_xa',
             'key_passes','successful_dribbles','dribbles_attempted','successful_passes','total_passes',
             'crosses','successful_crosses','tackles','interceptions','clearances','blocks','recoveries',
             'defensive_actions','aerial_duels_won','aerial_duels_lost','saves','clean_sheet',
             'goals_conceded','penalty_saves','yellow_cards','red_cards','fouls_committed',
             'fouls_suffered','offsides']

df_team = df.groupby('team')[rate_cols].mean().reset_index()

avg_extra = (df.groupby('team')[['pass_accuracy', 'save_percentage', 'market_value_eur']]
               .mean()
               .rename(columns={'pass_accuracy': 'avg_pass_accuracy',
                                 'save_percentage': 'avg_save_percentage',
                                 'market_value_eur': 'avg_market_value_eur'}))

df_team = df_team.merge(avg_extra, on='team')
print(f"df_team shape: {df_team.shape}  (expect 48 rows, one per team)")
df_team.head()


df_team shape: (48, 34)  (expect 48 rows, one per team)


,team,goals,assists,shots,shots_on_target,expected_goals_xg,expected_assists_xa,key_passes,successful_dribbles,dribbles_attempted,successful_passes,total_passes,crosses,successful_crosses,tackles,interceptions,clearances,blocks,recoveries,defensive_actions,aerial_duels_won,aerial_duels_lost,saves,clean_sheet,goals_conceded,penalty_saves,yellow_cards,red_cards,fouls_committed,fouls_suffered,offsides,avg_pass_accuracy,avg_save_percentage,avg_market_value_eur
0,Algeria,0.057223,0.048780,0.480300,0.051595,0.016248,0.019962,0.498124,0.163227,0.609756,15.353659,19.024390,0.407129,0.017824,0.804878,0.602251,0.832083,0.257036,1.362101,2.496248,0.779550,0.347092,0.111632,0.009381,0.045966,0.001876,0.088180,0.002814,0.416510,0.297373,0.093809,0.807195,0.025281,2.896488e+07
1,Argentina,0.052036,0.064480,0.484163,0.059955,0.018145,0.017364,0.476244,0.173077,0.621041,16.750000,20.703620,0.460407,0.019231,0.865385,0.621041,0.809955,0.228507,1.539593,2.524887,0.808824,0.359729,0.128959,0.009050,0.040724,0.000000,0.091629,0.004525,0.437783,0.322398,0.081448,0.808314,0.024966,2.412440e+07
2,Australia,0.049918,0.052373,0.463993,0.053191,0.018282,0.016702,0.509820,0.164484,0.611293,15.167758,18.851064,0.410802,0.009820,0.844517,0.650573,0.815876,0.210311,1.324059,2.521277,0.709493,0.376432,0.132570,0.006547,0.049100,0.001637,0.096563,0.005728,0.405074,0.360065,0.076105,0.810057,0.028273,2.138142e+07
3,Austria,0.062937,0.051282,0.469697,0.057110,0.019009,0.018368,0.523310,0.198135,0.689977,16.508159,20.538462,0.437063,0.019814,0.808858,0.657343,0.787879,0.229604,1.468531,2.483683,0.726107,0.396270,0.122378,0.008159,0.046620,0.002331,0.082751,0.006993,0.423077,0.325175,0.081585,0.808380,0.024977,2.009812e+07
4,Belgium,0.050528,0.051282,0.524133,0.066365,0.020671,0.019027,0.504525,0.189291,0.648567,15.581448,19.417044,0.446456,0.015083,0.788839,0.612368,0.797888,0.226998,1.407240,2.426094,0.719457,0.373303,0.127451,0.009050,0.051282,0.002262,0.105581,0.001508,0.404977,0.303922,0.082202,0.806433,0.026727,2.534505e+07


## 3. Correlation + VIF Check -- Narrowing Down to a Clean Feature Set

We rank every numeric column by its correlation with `avg_market_value_eur`, then check pairwise correlations
among the top candidates to catch any remaining redundant pairs (multicollinearity), and finally confirm with
VIF (Variance Inflation Factor).

**Important VIF gotcha:** `variance_inflation_factor()` gives meaningless, wildly inflated numbers if you don't
include an intercept/constant column in the matrix -- always run your features through `sm.add_constant()`
first.

In [19]:
target = 'avg_market_value_eur'
numeric_cols = [c for c in rate_cols] + ['avg_pass_accuracy', 'avg_save_percentage']

corr_with_target = df_team[numeric_cols + [target]].corr()[target].drop(target)
corr_with_target = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index)
print("Correlation with avg_market_value_eur, strongest first:")
print(corr_with_target.round(3))


Correlation with avg_market_value_eur, strongest first:
shots                  0.675
key_passes             0.636
shots_on_target        0.612
successful_passes      0.564
total_passes           0.563
successful_dribbles    0.538
dribbles_attempted     0.513
expected_assists_xa    0.422
expected_goals_xg      0.402
recoveries             0.385
goals                  0.354
assists                0.354
red_cards             -0.230
offsides              -0.188
avg_save_percentage    0.175
fouls_suffered        -0.167
blocks                 0.159
interceptions         -0.153
penalty_saves         -0.138
aerial_duels_lost      0.136
tackles                0.128
clean_sheet           -0.117
avg_pass_accuracy      0.105
fouls_committed        0.086
aerial_duels_won      -0.078
yellow_cards          -0.061
saves                 -0.052
goals_conceded         0.050
clearances            -0.043
successful_crosses    -0.029
crosses                0.020
defensive_actions      0.019
Name: avg_market

In [20]:
top_candidates = corr_with_target.head(13).index.tolist()
pairwise = df_team[top_candidates].corr()

high_pairs = []
for i in range(len(top_candidates)):
    for j in range(i + 1, len(top_candidates)):
        r = pairwise.iloc[i, j]
        if abs(r) > 0.85:
            high_pairs.append((top_candidates[i], top_candidates[j], round(r, 3)))

print(f"Pairs with |r| > 0.85 among the top {len(top_candidates)} correlates (these are redundant -- keep only one of each pair):")
for p in sorted(high_pairs, key=lambda x: -abs(x[2])):
    print(p)


Pairs with |r| > 0.85 among the top 13 correlates (these are redundant -- keep only one of each pair):
('successful_passes', 'total_passes', np.float64(0.998))
('successful_dribbles', 'dribbles_attempted', np.float64(0.963))
('shots', 'shots_on_target', np.float64(0.875))
('total_passes', 'recoveries', np.float64(0.87))
('successful_passes', 'recoveries', np.float64(0.864))


Based on that redundancy check, dropping `shots_on_target` (near-duplicate of `shots`), `total_passes`
(near-duplicate of `successful_passes`), and `dribbles_attempted` (near-duplicate of `successful_dribbles`)
leaves this clean set of 11 features. We confirm with VIF:

In [21]:
reg_features = ['shots', 'key_passes', 'successful_dribbles', 'successful_passes',
                'expected_assists_xa', 'expected_goals_xg', 'goals', 'assists',
                'avg_pass_accuracy', 'avg_save_percentage', 'red_cards']

df_team_reg = df_team[['team', target] + reg_features].copy()

X_check = sm.add_constant(df_team_reg[reg_features])
vif_table = pd.DataFrame({
    'feature': reg_features,
    'VIF': [variance_inflation_factor(X_check.values, i + 1) for i in range(len(reg_features))]
})
print("VIF for the final feature set (rule of thumb: VIF > 10 is concerning; all of these are well under it):")
print(vif_table.sort_values('VIF', ascending=False).to_string(index=False))


VIF for the final feature set (rule of thumb: VIF > 10 is concerning; all of these are well under it):
            feature      VIF
         key_passes 5.134074
successful_dribbles 4.502425
              shots 3.456147
  successful_passes 2.482203
  expected_goals_xg 1.989953
              goals 1.749328
expected_assists_xa 1.666817
            assists 1.654431
avg_save_percentage 1.277375
          red_cards 1.173477
  avg_pass_accuracy 1.164666


`df_team_reg` now holds `team`, the target (`avg_market_value_eur`), and 11 clean, low-collinearity
features. This is your starting point for the practice section below.

---

## 4. YOUR TURN -- Split into Two Feature Sets

We're going to build **two separate regressions** predicting the same target with two different, thematically
distinct feature groups, so we can compare which type of performance data explains market value better:

- **Model 1 -- "Attacking Output"**: `goals`, `assists`, `shots`, `expected_goals_xg`, `expected_assists_xa`,
  `successful_dribbles`
- **Model 2 -- "Possession & Discipline"**: `key_passes`, `successful_passes`, `avg_pass_accuracy`,
  `avg_save_percentage`, `red_cards`

Fill in the blank below (just column selection on `df_team_reg` -- same pattern as `df_attacking`, but with the
other feature list):

In [22]:
model1_features = ['goals', 'assists', 'shots', 'expected_goals_xg', 'expected_assists_xa', 'successful_dribbles']
model2_features = ['key_passes', 'successful_passes', 'avg_pass_accuracy', 'avg_save_percentage', 'red_cards']

df_attacking = df_team_reg[['team', target] + model1_features]
df_possession = df_team_reg[['team', target] + model2_features]  # TODO: same pattern as df_attacking, but using model2_features

print(df_attacking.shape)
print(df_possession.shape)


(48, 8)
(48, 7)


## 5. YOUR TURN -- Set Up `X` and `y`

Every scikit-learn regression needs two separate objects:

- **`X`** -- your input **features** (a DataFrame/array, 2D even if it's one column)
- **`y`** -- your **target**, the thing you're predicting (1D)

Fill in `y` below (it's a single column name, the same one for both models since they predict the same
target):

In [23]:
X_attacking = df_attacking[model1_features]
y_attacking = df_attacking[target]  # TODO: which single column are we trying to predict?

X_possession = df_possession[model2_features]
y_possession = df_possession[target]   # TODO: same column as above


## 6. YOUR TURN -- `train_test_split`

```python
from sklearn.model_selection import train_test_split
```

What this does: it **randomly shuffles and splits your rows** into a training group (the model learns from
these) and a test group (held out, never seen during fitting -- used only to check how well the model
generalizes to data it hasn't memorized).

- `test_size=0.2` -- 20% of rows go to the test set. With 48 teams that's roughly 9-10 held-out teams.
- `random_state=42` -- fixes the random shuffle so you get the *same* split every time you re-run the cell.
  Any integer works; it's a seed, not a meaningful number. Without it, results would differ run to run and
  wouldn't be reproducible.
- **Why split at all?** If you evaluate on the same rows you trained on, the model can partly "memorize" quirks
  of that exact data, making its reported accuracy optimistic. The held-out test set gives an honest estimate
  of how it'd perform on a team it has never seen -- which is the situation you're actually in when you use a
  model to predict something new.

Fill in the call for **both** feature sets (you'll end up with 8 variables total: 4 per model):

In [25]:
from sklearn.model_selection import train_test_split

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(____, ____, test_size=0.2, random_state=42)

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_attacking, y_attacking, test_size=0.2, random_state=42)  # TODO: same call, using X_possession / y_possession


ModuleNotFoundError: No module named 'sklearn'

## 7. YOUR TURN -- Fit the Model

```python
from sklearn.linear_model import LinearRegression
```

- `LinearRegression()` creates an **empty, untrained** model object -- instantiating it does no math yet.
- `.fit(X_train, y_train)` is the step that actually solves for the best-fit coefficients: it looks *only* at
  the training rows and finds the line/hyperplane that minimizes squared prediction error.
- After fitting, `model.coef_` gives the learned coefficient for each feature (same order as your `X` columns),
  and `model.intercept_` gives the constant term.

Fill in which two objects a model should `.fit()` on (hint: training data, not test data -- the model should
never see the test set until you're ready to evaluate it):

In [ ]:
from sklearn.linear_model import LinearRegression

model_attacking = LinearRegression()
model_attacking.fit(X_train_a, y_train_a)   # TODO: which X/y pair does fitting use?

model_possession = LinearRegression()
model_possession.fit(X_train_p, y_train_p)   # TODO: same idea, possession model's training data


## 8. YOUR TURN -- Predict and Evaluate

```python
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
```

- **`.predict(X)`** generates predicted values for whatever `X` you pass in. Think about which one you should
  pass here: if you want an *honest* measure of accuracy (not one the model can cheat on by having memorized
  the answers), which set should this be -- the one it trained on, or the one it's never seen?
- **`r2_score(y_true, y_pred)`** -- fraction of variance in the target your predictions explain. Pass in the
  true test values and your test-set predictions for an honest number.
- **`mean_absolute_error(y_true, y_pred)`** -- average size of your miss, in the same units as market value
  (euros). Easy to explain: "on average, off by about €X."
- **`mean_squared_error(y_true, y_pred)`**, then `np.sqrt(...)` to get RMSE -- similar, but penalizes large
  misses much more heavily than small ones. If RMSE is a lot bigger than MAE, a handful of teams are being
  predicted especially badly.

Fill in the blanks (there are two full sets -- one per model):

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# --- Attacking model ---
preds_attacking = model_attacking.predict(X_test_a)   # TODO: predict on which set?

r2_attacking = r2_score(y_test_a, preds_attacking)
mae_attacking = mean_absolute_error(y_test_a, preds_attacking)
rmse_attacking = np.sqrt(mean_squared_error(y_test_a, preds_attacking))

# --- Possession model ---
preds_possession = model_possession.predict(X_test_p)

r2_possession = r2_score(y_test_p, preds_possession)
mae_possession = mean_absolute_error(y_test_p, preds_possession)
rmse_possession = np.sqrt(mean_squared_error(y_test_p, preds_possession))

print(f"Attacking model   -- R2: {r2_attacking:.3f}  MAE: {mae_attacking:,.0f}  RMSE: {rmse_attacking:,.0f}")
print(f"Possession model  -- R2: {r2_possession:.3f}  MAE: {mae_possession:,.0f}  RMSE: {rmse_possession:,.0f}")


ModuleNotFoundError: No module named 'sklearn'

## 9. Compare and Interpret

Once both sets of numbers print above, think through:

- Which model has the higher **test R²**? That tells you which *type* of performance data (raw attacking
  output vs. possession/discipline) explains more of the variance in market value.
- Is **RMSE noticeably bigger than MAE** for either model? If so, a few teams are being predicted especially
  badly -- worth printing `y_test - preds` side by side with team names to see which ones.
- With only ~9-10 teams in each test set, **don't over-trust a single run** -- try changing `random_state` to a
  few different values and see whether the same model consistently comes out ahead, or whether the "winner"
  flips around depending on which teams happen to land in the test set. That instability is itself an
  important, honest thing to report if you were writing this up.